# 03 — Update Extract: Egyptian frames (fewer per speaker)

**Why this notebook:** The old extract (`colab_extract_modalink_frames.ipynb`) used ~**2 FPS** and up to **15 faces/segment**, so the same speaker flooded the dataset with near-duplicate frames. That hurts person-level fine-tuning (stuck on angry, weak TEST).

**What changed vs old extract**
- Lower sampling rate (`FPS_SAMPLE = 0.5` → about 1 frame every 2 seconds)
- Fewer faces per segment (`MAX_FACES_PER_SEGMENT = 6`)
- Hard cap per **person × emotion** (`MAX_FRAMES_PER_PERSON_EMOTION = 12`)
- Frames **spread across the clip** (not only the beginning)
- Optional blur filter
- Writes to a **new folder** so you keep the old frames: `egypt_modalink_frames_v3`

**Annotations:** `annotations version 2.xlsx`  
Label column: `Final Overall (majority of modalities)`  
Person key: `Folder|speaker`

**After this:** run Colab fine-tune `02_...` with `FRAMES_ROOT` pointing to the new folder.


In [ ]:
# 0) OpenCV 4.x (Colab OpenCV 5 breaks CascadeClassifier)
!pip uninstall -y opencv-python opencv-python-headless opencv-contrib-python opencv-contrib-python-headless
!pip install -q "opencv-python-headless==4.10.0.84" pandas openpyxl tqdm
print("Installed. Runtime → Restart session, then continue from next cell.")


Found existing installation: opencv-python 5.0.0.93
Uninstalling opencv-python-5.0.0.93:
  Successfully uninstalled opencv-python-5.0.0.93
Found existing installation: opencv-python-headless 5.0.0.93
Uninstalling opencv-python-headless-5.0.0.93:
  Successfully uninstalled opencv-python-headless-5.0.0.93
Found existing installation: opencv-contrib-python 4.13.0.92
Uninstalling opencv-contrib-python-4.13.0.92:
  Successfully uninstalled opencv-contrib-python-4.13.0.92
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 15.1 MB/s eta 0:00:00
Installed. Runtime → Restart session, then continue from next cell.


In [ ]:
# 1) Verify OpenCV (must be 4.x + CascadeClassifier)
import cv2
print("cv2 version:", cv2.__version__)
print("Has CascadeClassifier:", hasattr(cv2, "CascadeClassifier"))
assert str(cv2.__version__).startswith("4."), "Need OpenCV 4.x — restart runtime after cell 0"
assert hasattr(cv2, "CascadeClassifier"), "CascadeClassifier missing — reinstall OpenCV 4.10"


cv2 version: 4.10.0
Has CascadeClassifier: True


In [ ]:
# 2) Mount Drive
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

from pathlib import Path
print("MyDrive entries:")
for p in sorted(Path("/content/drive/MyDrive").iterdir())[:40]:
    print(" -", p.name)


Mounted at /content/drive
MyDrive entries:
 - Colab Notebooks
 - Final Modalink Dataset
 - MasterData
 - egypt_modalink_frames


In [ ]:
# 3) Paths — EDIT if your shortcut names differ
from pathlib import Path

MYDRIVE = Path("/content/drive/MyDrive")

# Source videos (Add shortcut of Final Modalink Dataset into My Drive)
MODALINK_ROOT = MYDRIVE / "Final Modalink Dataset"

# NEW output folder (does not overwrite old egypt_modalink_frames)
FRAMES_ROOT = MYDRIVE / "MasterData" / "egypt_modalink_frames_v3"

# Annotations: upload below OR put on Drive
ANNOTATIONS_XLSX = Path("/content/annotations_version_2.xlsx")

FRAMES_ROOT.mkdir(parents=True, exist_ok=True)
print("MODALINK_ROOT exists:", MODALINK_ROOT.exists(), "→", MODALINK_ROOT)
print("FRAMES_ROOT:", FRAMES_ROOT)

if not MODALINK_ROOT.exists():
    print("ERROR: add shortcut 'Final Modalink Dataset' to My Drive, then re-run.")
    for p in MYDRIVE.rglob("*"):
        if p.is_dir() and "modalink" in p.name.lower():
            print(" candidate:", p)


MODALINK_ROOT exists: True → /content/drive/MyDrive/Final Modalink Dataset
FRAMES_ROOT: /content/drive/MyDrive/MasterData/egypt_modalink_frames_v3


In [ ]:
# 4) Load annotations Excel (annotations version 2.xlsx)
from google.colab import files
import pandas as pd

if not ANNOTATIONS_XLSX.exists():
    candidates = list(MYDRIVE.rglob("*annotations*version*2*.xlsx"))
    candidates += list(MYDRIVE.rglob("*annotations version 2.xlsx"))
    if candidates:
        ANNOTATIONS_XLSX = candidates[0]
        print("Found on Drive:", ANNOTATIONS_XLSX)
    else:
        print("Upload annotations version 2.xlsx ...")
        uploaded = files.upload()
        name = next(iter(uploaded))
        ANNOTATIONS_XLSX = Path("/content") / name
        print("Uploaded:", ANNOTATIONS_XLSX)
else:
    print("Using:", ANNOTATIONS_XLSX)

df = pd.read_excel(ANNOTATIONS_XLSX)
print("Rows:", len(df))
print("Columns:", list(df.columns))
LABEL_COL = "Final Overall (majority of modalities)"
print("\nLabel counts:")
print(df[LABEL_COL].value_counts(dropna=False))
print("\nUnique Folder:", df["Folder"].nunique(), "| speakers:", df["speaker"].nunique())
df.head(3)


Upload annotations version 2.xlsx ...


Saving annotations version 2.xlsx to annotations version 2.xlsx
Uploaded: /content/annotations version 2.xlsx
Rows: 956
Columns: ['Folder', 'segment_id', 'speaker', 'speaker_segment_id', 'start_time', 'end_time', 'duration', 'video_file', 'audio_file', 'transcript', 'Emotion Audio (final)', 'Text Emotion (final)', 'Video Emotion (final)', 'Final Overall (majority of modalities)', 'Audio Clarity (final)', 'Speaker Gender (any)', 'Speaker Identity (any)']

Label counts:
Final Overall (majority of modalities)
Anger        268
Sadness      198
Neutral      172
Disgust      116
Surprise      77
Happiness     75
Fear          50
Name: count, dtype: int64

Unique Folder: 108 | speakers: 9


,Folder,segment_id,speaker,speaker_segment_id,start_time,end_time,duration,video_file,audio_file,transcript,Emotion Audio (final),Text Emotion (final),Video Emotion (final),Final Overall (majority of modalities),Audio Clarity (final),Speaker Gender (any),Speaker Identity (any)
0,videoplayback (1),3,SPEAKER_00,0,126.981594,140.937219,13.955625,videos/SPEAKER_00/SPEAKER_00_segment_0000.mp4,audios/SPEAKER_00/SPEAKER_00_segment_0000.wav,أنا مش مصدقة إن إنت خايف منها قوي كدا إنت لازم...,Anger,Sadness,Anger,Anger,Clear,Female,NaN
1,videoplayback (1),4,SPEAKER_00,1,152.142219,166.772844,14.630625,videos/SPEAKER_00/SPEAKER_00_segment_0001.mp4,audios/SPEAKER_00/SPEAKER_00_segment_0001.wav,كل ده بقى عشان انت شايفني ضعيفة ومقدرش أعيش من...,NaN,Anger,NaN,Anger,Clear,Female,NaN
2,videoplayback (10),0,SPEAKER_01,0,0.030969,29.325969,29.295000,videos/SPEAKER_01/SPEAKER_01_segment_0000.mp4,audios/SPEAKER_01/SPEAKER_01_segment_0000.wav,كان قبلها مات الله ارحمه مصطفى المتولي وده كان...,NaN,Sadness,NaN,Sadness,Clear,Male,NaN


In [ ]:
# 5) Config — fewer / cleaner frames per speaker
import re
import cv2
import numpy as np
from collections import defaultdict
from tqdm.auto import tqdm

EMOTION_MAP = {
    "Anger": "angry",
    "Disgust": "disgust",
    "Fear": "fear",
    "Happiness": "happy",
    "Neutral": "neutral",
    "Sadness": "sad",
    "Surprise": "surprise",
}
SKIP_LABELS = {"", "nan", "none", "ambiguous", "amiguous"}

# --- sampling (UPDATED vs old extract) ---
FPS_SAMPLE = 0.5                 # ~1 frame every 2 seconds (old was 2.0)
MAX_FACES_PER_SEGMENT = 6        # old was 15
MAX_FRAMES_PER_PERSON_EMOTION = 12  # hard global cap for each Folder|speaker × emotion
MIN_FACE_SIZE = 70
FACE_PAD = 0.25
SAVE_SIZE = 96
BLUR_VAR_MIN = 40.0              # drop very blurry faces (Laplacian variance)
DRY_RUN_LIMIT = None             # e.g. 25 for a quick path test; None = all

CASCADE = cv2.CascadeClassifier(
    str(Path(cv2.data.haarcascades) / "haarcascade_frontalface_default.xml")
)
assert not CASCADE.empty(), "Haar cascade failed to load"

print("FPS_SAMPLE:", FPS_SAMPLE)
print("MAX_FACES_PER_SEGMENT:", MAX_FACES_PER_SEGMENT)
print("MAX_FRAMES_PER_PERSON_EMOTION:", MAX_FRAMES_PER_PERSON_EMOTION)


FPS_SAMPLE: 0.5
MAX_FACES_PER_SEGMENT: 6
MAX_FRAMES_PER_PERSON_EMOTION: 12


In [ ]:
# 6) Build worklist from Excel
def normalize_label(raw):
    if pd.isna(raw):
        return None
    s = str(raw).strip()
    if s.lower() in SKIP_LABELS:
        return None
    return EMOTION_MAP.get(s)

def resolve_video_path(folder: str, video_file: str):
    rel = Path(str(folder)) / str(video_file)
    full = MODALINK_ROOT / rel
    if full.exists():
        return full
    folder_dir = MODALINK_ROOT / str(folder)
    name = Path(str(video_file)).name
    if folder_dir.exists():
        hits = list(folder_dir.rglob(name))
        if hits:
            return hits[0]
    return None

rows = []
skipped = {"no_label": 0, "missing_video": 0}

for i, r in df.iterrows():
    emotion = normalize_label(r.get(LABEL_COL))
    if emotion is None:
        skipped["no_label"] += 1
        continue
    folder = str(r["Folder"]).strip()
    video_file = str(r["video_file"]).strip()
    speaker = str(r["speaker"]).strip()
    person_id = f"{folder}|{speaker}"
    vpath = resolve_video_path(folder, video_file)
    if vpath is None:
        skipped["missing_video"] += 1
        continue
    rows.append({
        "excel_row": int(i),
        "folder": folder,
        "speaker": speaker,
        "person_id": person_id,
        "emotion": emotion,
        "video_path": str(vpath),
        "segment_id": r.get("segment_id"),
        "speaker_segment_id": r.get("speaker_segment_id"),
        "duration": float(r.get("duration") or 0.0),
    })

work = pd.DataFrame(rows)
# Prefer longer/diverse coverage: shuffle then we'll enforce caps while iterating
work = work.sample(frac=1.0, random_state=42).reset_index(drop=True)
if DRY_RUN_LIMIT:
    work = work.head(DRY_RUN_LIMIT)

print("Usable segments:", len(work))
print("Skipped:", skipped)
print("Emotion counts (segments):\n", work["emotion"].value_counts())
print("Unique persons:", work["person_id"].nunique())
work.head(3)


Usable segments: 956
Skipped: {'no_label': 0, 'missing_video': 0}
Emotion counts (segments):
 emotion
angry       268
sad         198
neutral     172
disgust     116
surprise     77
happy        75
fear         50
Name: count, dtype: int64
Unique persons: 220


,excel_row,folder,speaker,person_id,emotion,video_path,segment_id,speaker_segment_id,duration
0,342,videoplayback (141),SPEAKER_01,videoplayback (141)|SPEAKER_01,disgust,/content/drive/MyDrive/Final Modalink Dataset/...,9,4,3.51000
1,864,videoplayback (81),SPEAKER_02,videoplayback (81)|SPEAKER_02,neutral,/content/drive/MyDrive/Final Modalink Dataset/...,24,18,7.35750
2,522,videoplayback (225),SPEAKER_01,videoplayback (225)|SPEAKER_01,fear,/content/drive/MyDrive/Final Modalink Dataset/...,5,3,5.36625


In [ ]:
# 7) Face helpers + spread sampling across the clip
def is_blurry(face_bgr, thr=BLUR_VAR_MIN) -> bool:
    gray = cv2.cvtColor(face_bgr, cv2.COLOR_BGR2GRAY)
    return float(cv2.Laplacian(gray, cv2.CV_64F).var()) < thr

def largest_face_bgr(frame_bgr):
    gray = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY)
    faces = CASCADE.detectMultiScale(
        gray, scaleFactor=1.1, minNeighbors=5, minSize=(MIN_FACE_SIZE, MIN_FACE_SIZE)
    )
    if len(faces) == 0:
        return None
    x, y, w, h = max(faces, key=lambda f: f[2] * f[3])
    pad_x, pad_y = int(w * FACE_PAD), int(h * FACE_PAD)
    x1, y1 = max(0, x - pad_x), max(0, y - pad_y)
    x2 = min(frame_bgr.shape[1], x + w + pad_x)
    y2 = min(frame_bgr.shape[0], y + h + pad_y)
    return frame_bgr[y1:y2, x1:x2]

def safe_stem(person_id: str, emotion: str, segment_id) -> str:
    pid = re.sub(r"[^A-Za-z0-9_|-]+", "_", person_id)
    return f"{pid}__{emotion}__seg{segment_id}"

def candidate_frame_indices(n_frames: int, fps: float, max_keep: int) -> list[int]:
    """Spread sample indices across the whole video (not only the start)."""
    if n_frames <= 0 or max_keep <= 0:
        return []
    # desired temporal step from FPS_SAMPLE
    step = max(1, int(round(fps / max(FPS_SAMPLE, 1e-6))))
    # Build evenly spaced pool, then thin to ~max_keep*3 candidates for face detect
    idxs = list(range(0, n_frames, step))
    if not idxs:
        return [0]
    # Always include near start / mid / end if possible
    extras = {0, n_frames // 2, max(0, n_frames - 1)}
    idxs = sorted(set(idxs) | extras)
    # If still too many, evenly subsample candidate list
    oversample = max(max_keep * 3, max_keep)
    if len(idxs) > oversample:
        sel = np.linspace(0, len(idxs) - 1, num=oversample)
        idxs = [idxs[int(i)] for i in sel]
    return idxs

def extract_faces_from_video(video_path: Path, out_dir: Path, stem: str, max_keep: int) -> list[str]:
    out_dir.mkdir(parents=True, exist_ok=True)
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return []

    fps = float(cap.get(cv2.CAP_PROP_FPS) or 25.0)
    n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    targets = candidate_frame_indices(n_frames, fps, max_keep)

    saved = []
    for fi in targets:
        if len(saved) >= max_keep:
            break
        cap.set(cv2.CAP_PROP_POS_FRAMES, fi)
        ok, frame = cap.read()
        if not ok:
            continue
        face = largest_face_bgr(frame)
        if face is None or face.size == 0:
            continue
        if is_blurry(face):
            continue
        face = cv2.resize(face, (SAVE_SIZE, SAVE_SIZE), interpolation=cv2.INTER_AREA)
        out_path = out_dir / f"{stem}__f{len(saved):03d}.jpg"
        cv2.imwrite(str(out_path), face, [int(cv2.IMWRITE_JPEG_QUALITY), 95])
        saved.append(str(out_path))

    cap.release()
    return saved

print("Helpers ready.")


Helpers ready.


In [ ]:
# 8) Run extraction with per-person×emotion budget
import json

manifest_rows = []
n_images = 0
n_empty = 0
n_skipped_budget = 0
budget = defaultdict(int)  # (person_id, emotion) -> frames kept

for _, row in tqdm(work.iterrows(), total=len(work)):
    emotion = row["emotion"]
    person_id = row["person_id"]
    key = (person_id, emotion)
    remaining = MAX_FRAMES_PER_PERSON_EMOTION - budget[key]
    if remaining <= 0:
        n_skipped_budget += 1
        continue

    max_keep = min(MAX_FACES_PER_SEGMENT, remaining)
    out_dir = FRAMES_ROOT / "by_emotion" / emotion / person_id.replace("|", "__")
    stem = safe_stem(person_id, emotion, row["speaker_segment_id"])
    paths = extract_faces_from_video(Path(row["video_path"]), out_dir, stem, max_keep=max_keep)

    if not paths:
        n_empty += 1
        continue

    budget[key] += len(paths)
    for p in paths:
        manifest_rows.append({
            "image_path": p,
            "emotion": emotion,
            "person_id": person_id,
            "folder": row["folder"],
            "speaker": row["speaker"],
            "video_path": row["video_path"],
            "excel_row": row["excel_row"],
        })
        n_images += 1

manifest = pd.DataFrame(manifest_rows)
manifest_path = FRAMES_ROOT / "manifest.csv"
manifest.to_csv(manifest_path, index=False)

# Budget stats
budget_df = pd.DataFrame(
    [{"person_id": k[0], "emotion": k[1], "n_frames": v} for k, v in budget.items()]
)

summary = {
    "n_segments_attempted": int(len(work)),
    "n_segments_skipped_budget": int(n_skipped_budget),
    "n_segments_no_face": int(n_empty),
    "n_images": int(n_images),
    "emotion_counts": manifest["emotion"].value_counts().to_dict() if len(manifest) else {},
    "n_persons": int(manifest["person_id"].nunique()) if len(manifest) else 0,
    "max_frames_per_person_emotion": MAX_FRAMES_PER_PERSON_EMOTION,
    "max_faces_per_segment": MAX_FACES_PER_SEGMENT,
    "fps_sample": FPS_SAMPLE,
    "label_column": LABEL_COL,
    "frames_root": str(FRAMES_ROOT),
    "mean_frames_per_person_emotion": float(budget_df["n_frames"].mean()) if len(budget_df) else 0,
    "max_frames_observed": int(budget_df["n_frames"].max()) if len(budget_df) else 0,
}
with open(FRAMES_ROOT / "extract_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))
print("Manifest:", manifest_path)
if len(budget_df):
    print("\nFrames per person×emotion (head):")
    print(budget_df.sort_values("n_frames", ascending=False).head(10))


  0%|          | 0/956 [00:00<?, ?it/s]

{
  "n_segments_attempted": 956,
  "n_segments_skipped_budget": 317,
  "n_segments_no_face": 96,
  "n_images": 2135,
  "emotion_counts": {
    "angry": 586,
    "sad": 490,
    "surprise": 252,
    "happy": 249,
    "neutral": 226,
    "disgust": 209,
    "fear": 123
  },
  "n_persons": 199,
  "max_frames_per_person_emotion": 12,
  "max_faces_per_segment": 6,
  "fps_sample": 0.5,
  "label_column": "Final Overall (majority of modalities)",
  "frames_root": "/content/drive/MyDrive/MasterData/egypt_modalink_frames_v3",
  "mean_frames_per_person_emotion": 5.980392156862745,
  "max_frames_observed": 12
}
Manifest: /content/drive/MyDrive/MasterData/egypt_modalink_frames_v3/manifest.csv

Frames per person×emotion (head):
                          person_id   emotion  n_frames
0    videoplayback (141)|SPEAKER_01   disgust        12
291   videoplayback (67)|SPEAKER_01       sad        12
59    videoplayback (71)|SPEAKER_04     angry        12
309   videoplayback (19)|SPEAKER_00  surprise       

## Output

```text
MasterData/egypt_modalink_frames_v3/
  by_emotion/{emotion}/{person}/...jpg
  manifest.csv
  extract_summary.json
```

## Next: fine-tune (notebook 02)
In `02_colab_efficientnet_egypt_finetune.ipynb` CONFIG cell, set:

```python
FRAMES_ROOT = MYDRIVE / "MasterData" / "egypt_modalink_frames_v3"
```

Keep baseline:

```python
BASELINE_H5 = Path("/content/best_model_efficientnet_b0.h5")
```

Then run 02 as usual (person-level split + train). Compare TEST macro-F1 to v2 (~0.23).

### Tunable knobs (cell 5)
| Knob | Default | Meaning |
|------|---------|---------|
| `FPS_SAMPLE` | 0.5 | lower = fewer temporal duplicates |
| `MAX_FACES_PER_SEGMENT` | 6 | max from one clip |
| `MAX_FRAMES_PER_PERSON_EMOTION` | 12 | hard cap per speaker×emotion |
| `BLUR_VAR_MIN` | 40 | higher = stricter sharpness |
